<a href="https://colab.research.google.com/github/SheethHassan/AI-Based-Pneumonia-Detection/blob/ML-MODEL/Pneumonia_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#PYTHON LIBRARIES

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, callbacks, applications, models
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score)

In [3]:
# LOADING DATASET
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#Verfing Folder Structure

import os
for split in ['train', 'val', 'test']:
  for cls in ['NORMAL', 'PNEUMONIA']:
    path = f'/content/drive/MyDrive/chest_xray/chest_xray/{split}/{cls}'
    count = len(os.listdir(path))
    print(f"{split}/{cls}:{count} images")

train/NORMAL:1342 images
train/PNEUMONIA:3876 images
val/NORMAL:9 images
val/PNEUMONIA:9 images
test/NORMAL:234 images
test/PNEUMONIA:390 images


In [4]:
#Fix Valadation (since val is only 9 images)

base_dir = 'content/chest_xray/chest_xray'
train_dir = f'{base_dir}/train'
val_dir = f'{base_dir}/val'
test_dir = f'{base_dir}test'

In [5]:
#Constants

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
INPUT_SHAPE = (224,224,3)
SEED = 42

In [6]:

# Fix the directory paths
base_dir = '/content/drive/MyDrive/chest_xray/chest_xray'
train_dir = f'{base_dir}/train'
test_dir = f'{base_dir}/test'

#PREPROCESSING & AUGMENTATION
#AUGMENTATION ONLY FOR TRAINING DATA
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 20,
    width_shift_range = 0.1,
    height_shift_range = 0.1,
    shear_range = 0.15,
    zoom_range = 0.2,
    horizontal_flip = True,
    brightness_range = [0.8, 1.2],
    fill_mode = "nearest",
    validation_split = 0.20 #validation split
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = 'binary',
    seed = SEED,
    shuffle = True,
    subset = 'training'
)
val_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = "binary",
    seed = SEED,
    classes = CLASS_NAMES,
    subset = "validation"

)
test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = "binary",
    shuffle = False
)

print(f"Training Images: {train_gen.samples}")
print(f"validation Images:{val_gen.samples}")
print(f"Test Images: {test_gen.samples}")
print(f"Class indicies: {train_gen.class_indices}")

Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Training Images: 4173
validation Images:1043
Test Images: 624
Class indicies: {'NORMAL': 0, 'PNEUMONIA': 1}


In [7]:
#CLASS IMBALANCE
#Since the dataset has 1341 images train/NORMAL and 3875 images train/PNEUMONIA during traing without correction the model learns a bad habit which is guessing pneumonia 3x than guessing normal images which is called
# a biased model since it will miss real normal cases.
# This is fixed by using class imabalance
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight = 'balanced',
    classes = np.array([0,1]),
    y = train_gen.classes
)
class_weight_dict = {0:class_weights[0], 1:class_weights[1]}

print(f"Class weight for NORMAL is: {class_weight_dict[0]:.4f}")
print(f"Class weight for PNEUMONIA is : {class_weight_dict[1]:.4f}")

Class weight for NORMAL is: 1.9445
Class weight for PNEUMONIA is : 0.6731


In [8]:
#MOBILENETv2 MODEL
#Load Mobilenetv2 pretrained on ImageNet
base_model = MobileNetV2(
    input_shape = INPUT_SHAPE,
    include_top = False, #Remove Imagenet classification head
    weights = "imagenet" #use pretrained weights
)

#Freeze Model

base_model.trainable = False


#Main Model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation = 'relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid') #Binary output either PNEUMONIA OR NOT
])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
# MODEL COMPILE

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.0001),
    loss = 'binary_crossentropy',
    metrics = [
        'accuracy',
        tf.keras.metrics.AUC(name = 'auc'),
        tf.keras.metrics.Precision(name = 'precision'),
        tf.keras.metrics.Recall(name = 'recall')
    ]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,435,393 (9.29 MB)

 Trainable params: 174,849 (683.00 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

In [10]:
#Callbacks
callbacks = [
    #stop learning if val_loss doesn't improve in 5 epochs
    tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        patience = 5,
        restore_best_weights = True,
        verbose = 1
    ),

    #reduce learning rate if val_loss plateaus for 3 epochs
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor = 'val_loss',
        factor = 0.5,
        patience = 3,
        verbose = 1,
        min_lr = 0.0000001
    ),
    #save the best model automatically
    tf.keras.callbacks.ModelCheckpoint(
        filepath = '/content/drive/MyDrive/best_model.h5',
        monitor = 'val_auc',
        save_best_only = True,
        verbose = 1
    )
]

#Phase 1 -- Custom Top Layer Training
print("=" * 50)
print("Phase 1: Training Top layers of the model only")
print(f"Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")
print("=" * 50)

history_phase1 = model.fit(
    train_gen,
    validation_data = val_gen,
    epochs = 20,
    class_weight = class_weight_dict,
    callbacks = callbacks
)

Phase 1: Training Top layers of the model only
Trainable params: 174,849
Epoch 1/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.5979 - auc: 0.7690 - loss: 0.5953 - precision: 0.9015 - recall: 0.5039
Epoch 1: val_auc improved from None to 0.95977, saving model to /content/drive/MyDrive/best_model.h5



Epoch 1: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 1400s 11s/step - accuracy: 0.7304 - auc: 0.8807 - loss: 0.4520 - precision: 0.9507 - recall: 0.6719 - val_accuracy: 0.8773 - val_auc: 0.9598 - val_loss: 0.3373 - val_precision: 0.9709 - val_recall: 0.8606 - learning_rate: 1.0000e-04
Epoch 2/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8782 - auc: 0.9548 - loss: 0.2840 - precision: 0.9688 - recall: 0.8626
Epoch 2: val_auc improved from 0.95977 to 0.96753, saving model to /content/drive/MyDrive/best_model.h5



Epoch 2: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 387s 3s/step - accuracy: 0.8840 - auc: 0.9568 - loss: 0.2707 - precision: 0.9729 - recall: 0.8681 - val_accuracy: 0.8802 - val_auc: 0.9675 - val_loss: 0.2879 - val_precision: 0.9808 - val_recall: 0.8555 - learning_rate: 1.0000e-04
Epoch 3/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8847 - auc: 0.9650 - loss: 0.2391 - precision: 0.9714 - recall: 0.8735
Epoch 3: val_auc improved from 0.96753 to 0.97784, saving model to /content/drive/MyDrive/best_model.h5



Epoch 3: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 386s 3s/step - accuracy: 0.8919 - auc: 0.9669 - loss: 0.2322 - precision: 0.9705 - recall: 0.8813 - val_accuracy: 0.8984 - val_auc: 0.9778 - val_loss: 0.2362 - val_precision: 0.9841 - val_recall: 0.8774 - learning_rate: 1.0000e-04
Epoch 4/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9131 - auc: 0.9746 - loss: 0.2023 - precision: 0.9748 - recall: 0.9071
Epoch 4: val_auc did not improve from 0.97784
131/131 ━━━━━━━━━━━━━━━━━━━━ 385s 3s/step - accuracy: 0.9092 - auc: 0.9726 - loss: 0.2097 - precision: 0.9735 - recall: 0.9023 - val_accuracy: 0.9070 - val_auc: 0.9769 - val_loss: 0.2283 - val_precision: 0.9829 - val_recall: 0.8903 - learning_rate: 1.0000e-04
Epoch 5/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9169 - auc: 0.9750 - loss: 0.1983 - precision: 0.9760 - recall: 0.9118
Epoch 5: val_auc improved from 0.97784 to 0.98328, saving model to /content/drive/MyDrive/


Epoch 5: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 391s 3s/step - accuracy: 0.9200 - auc: 0.9765 - loss: 0.1933 - precision: 0.9766 - recall: 0.9142 - val_accuracy: 0.9080 - val_auc: 0.9833 - val_loss: 0.2316 - val_precision: 0.9942 - val_recall: 0.8813 - learning_rate: 1.0000e-04
Epoch 6/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9144 - auc: 0.9748 - loss: 0.1993 - precision: 0.9804 - recall: 0.9042
Epoch 6: val_auc did not improve from 0.98328
131/131 ━━━━━━━━━━━━━━━━━━━━ 493s 3s/step - accuracy: 0.9185 - auc: 0.9753 - loss: 0.1971 - precision: 0.9772 - recall: 0.9116 - val_accuracy: 0.9147 - val_auc: 0.9787 - val_loss: 0.2213 - val_precision: 0.9886 - val_recall: 0.8955 - learning_rate: 1.0000e-04
Epoch 7/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9235 - auc: 0.9789 - loss: 0.1843 - precision: 0.9768 - recall: 0.9192
Epoch 7: val_auc did not improve from 0.98328
131/131 ━━━━━━━━━━━━━━━━━━━━ 385s 3s/step - 

...


Epoch 10: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 383s 3s/step - accuracy: 0.9363 - auc: 0.9831 - loss: 0.1594 - precision: 0.9807 - recall: 0.9326 - val_accuracy: 0.9406 - val_auc: 0.9870 - val_loss: 0.1614 - val_precision: 0.9904 - val_recall: 0.9290 - learning_rate: 1.0000e-04
Epoch 11/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9366 - auc: 0.9842 - loss: 0.1557 - precision: 0.9803 - recall: 0.9334
Epoch 11: val_auc improved from 0.98696 to 0.98811, saving model to /content/drive/MyDrive/best_model.h5



Epoch 11: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 392s 3s/step - accuracy: 0.9348 - auc: 0.9823 - loss: 0.1658 - precision: 0.9783 - recall: 0.9329 - val_accuracy: 0.9214 - val_auc: 0.9881 - val_loss: 0.1901 - val_precision: 0.9901 - val_recall: 0.9032 - learning_rate: 1.0000e-04
Epoch 12/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9356 - auc: 0.9831 - loss: 0.1596 - precision: 0.9820 - recall: 0.9300
Epoch 12: val_auc improved from 0.98811 to 0.98905, saving model to /content/drive/MyDrive/best_model.h5



Epoch 12: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 381s 3s/step - accuracy: 0.9343 - auc: 0.9836 - loss: 0.1581 - precision: 0.9826 - recall: 0.9281 - val_accuracy: 0.9415 - val_auc: 0.9890 - val_loss: 0.1512 - val_precision: 0.9890 - val_recall: 0.9316 - learning_rate: 1.0000e-04
Epoch 13/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9404 - auc: 0.9845 - loss: 0.1532 - precision: 0.9820 - recall: 0.9375
Epoch 13: val_auc did not improve from 0.98905
131/131 ━━━━━━━━━━━━━━━━━━━━ 392s 3s/step - accuracy: 0.9387 - auc: 0.9833 - loss: 0.1596 - precision: 0.9817 - recall: 0.9348 - val_accuracy: 0.9319 - val_auc: 0.9883 - val_loss: 0.1704 - val_precision: 0.9916 - val_recall: 0.9161 - learning_rate: 1.0000e-04
Epoch 14/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9445 - auc: 0.9867 - loss: 0.1428 - precision: 0.9847 - recall: 0.9396
Epoch 14: val_auc did not improve from 0.98905
131/131 ━━━━━━━━━━━━━━━━━━━━ 390s 3s/st


Epoch 16: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 392s 3s/step - accuracy: 0.9478 - auc: 0.9877 - loss: 0.1309 - precision: 0.9901 - recall: 0.9390 - val_accuracy: 0.9453 - val_auc: 0.9909 - val_loss: 0.1302 - val_precision: 0.9825 - val_recall: 0.9432 - learning_rate: 5.0000e-05
Epoch 17/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9337 - auc: 0.9839 - loss: 0.1561 - precision: 0.9808 - recall: 0.9295
Epoch 17: val_auc improved from 0.99085 to 0.99151, saving model to /content/drive/MyDrive/best_model.h5



Epoch 17: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 386s 3s/step - accuracy: 0.9370 - auc: 0.9830 - loss: 0.1625 - precision: 0.9791 - recall: 0.9352 - val_accuracy: 0.9367 - val_auc: 0.9915 - val_loss: 0.1431 - val_precision: 0.9876 - val_recall: 0.9265 - learning_rate: 5.0000e-05
Epoch 18/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9394 - auc: 0.9874 - loss: 0.1429 - precision: 0.9819 - recall: 0.9359
Epoch 18: val_auc did not improve from 0.99151
131/131 ━━━━━━━━━━━━━━━━━━━━ 384s 3s/step - accuracy: 0.9430 - auc: 0.9875 - loss: 0.1393 - precision: 0.9841 - recall: 0.9384 - val_accuracy: 0.9319 - val_auc: 0.9863 - val_loss: 0.1786 - val_precision: 0.9889 - val_recall: 0.9187 - learning_rate: 5.0000e-05
Epoch 19/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9506 - auc: 0.9887 - loss: 0.1305 - precision: 0.9869 - recall: 0.9456
Epoch 19: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.

Epoch 19

In [ ]:
#PHASE 2 - Fine Tuning the Model
# Unfreeze Stage of top layers and training the rest of the model

print("=" * 50)
print("PHASE 2: Fine Tuning")
print("=" * 50)

#Unfreezing Model
base_model.trainable = True

#Freeze all layers except last 30 layers
for layer in base_model.layers[:-30]:
  layer.trainable = False


#Check how many layres are now trainable
trainable_layers = sum(1 for layer in base_model.layers if layer.trainable)
print(f"Trainable layers: {trainable_layers}")

#Recompile with a much lower rate to increase accuracy a bit higher
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.00001),
    loss = 'binary_crossentropy',
    metrics = [
        'accuracy',
        tf.keras.metrics.AUC(name = 'auc'),
        tf.keras.metrics.Precision(name = 'precision'),
        tf.keras.metrics.Recall(name = 'recall')
    ]
)

#update callbacks for phase 2
callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        patience = 5,
        restore_best_weights = True,
        verbose = 1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor = 'val_loss',
        factor = 0.5,
        patience = 3,
        verbose = 1,
        min_lr = 0.0000001
    ),
    #Save best model in google drive
    tf.keras.callbacks.ModelCheckpoint(
        filepath = '/content/drive/MyDrive/best_model.h5',
        monitor = 'val_auc',
        save_best_only = True,
        verbose = 1
    )
]

#Train Second Phase
history_phase2 = model.fit(
    train_gen,
    validation_data = val_gen,
    epochs = 20,
    class_weight = class_weight_dict,
    callbacks = callbacks_phase2
)


PHASE 2: Fine Tuning
Trainable layers: 30
Epoch 1/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9097 - auc: 0.9599 - loss: 0.2883 - precision: 0.9398 - recall: 0.9381
Epoch 1: val_auc improved from None to 0.98708, saving model to /content/drive/MyDrive/best_model.h5



Epoch 1: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 470s 3s/step - accuracy: 0.9085 - auc: 0.9611 - loss: 0.2679 - precision: 0.9524 - recall: 0.9229 - val_accuracy: 0.9271 - val_auc: 0.9871 - val_loss: 0.1861 - val_precision: 0.9176 - val_recall: 0.9910 - learning_rate: 1.0000e-05
Epoch 2/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9227 - auc: 0.9773 - loss: 0.1920 - precision: 0.9773 - recall: 0.9180
Epoch 2: val_auc did not improve from 0.98708
131/131 ━━━━━━━━━━━━━━━━━━━━ 454s 3s/step - accuracy: 0.9236 - auc: 0.9750 - loss: 0.2020 - precision: 0.9741 - recall: 0.9216 - val_accuracy: 0.9032 - val_auc: 0.9608 - val_loss: 0.3183 - val_precision: 0.8874 - val_recall: 0.9961 - learning_rate: 1.0000e-05
Epoch 3/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9251 - auc: 0.9779 - loss: 0.1895 - precision: 0.9774 - recall: 0.9205
Epoch 3: val_auc did not improve from 0.98708
131/131 ━━━━━━━━━━━━━━━━━━━━ 441s 3s/step - 


Epoch 8: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 448s 3s/step - accuracy: 0.9377 - auc: 0.9858 - loss: 0.1461 - precision: 0.9830 - recall: 0.9323 - val_accuracy: 0.9501 - val_auc: 0.9881 - val_loss: 0.1262 - val_precision: 0.9491 - val_recall: 0.9858 - learning_rate: 5.0000e-06
Epoch 9/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9479 - auc: 0.9863 - loss: 0.1416 - precision: 0.9863 - recall: 0.9424
Epoch 9: val_auc did not improve from 0.98811
131/131 ━━━━━━━━━━━━━━━━━━━━ 503s 3s/step - accuracy: 0.9473 - auc: 0.9873 - loss: 0.1356 - precision: 0.9855 - recall: 0.9429 - val_accuracy: 0.9511 - val_auc: 0.9871 - val_loss: 0.1231 - val_precision: 0.9559 - val_recall: 0.9794 - learning_rate: 5.0000e-06
Epoch 10/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9453 - auc: 0.9894 - loss: 0.1248 - precision: 0.9883 - recall: 0.9377
Epoch 10: val_auc improved from 0.98811 to 0.98854, saving model to /content/drive/MyDriv


Epoch 10: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 452s 3s/step - accuracy: 0.9434 - auc: 0.9881 - loss: 0.1346 - precision: 0.9844 - recall: 0.9387 - val_accuracy: 0.9540 - val_auc: 0.9885 - val_loss: 0.1178 - val_precision: 0.9642 - val_recall: 0.9742 - learning_rate: 5.0000e-06
Epoch 11/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9461 - auc: 0.9893 - loss: 0.1276 - precision: 0.9852 - recall: 0.9414
Epoch 11: val_auc improved from 0.98854 to 0.99279, saving model to /content/drive/MyDrive/best_model.h5



Epoch 11: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 449s 3s/step - accuracy: 0.9410 - auc: 0.9880 - loss: 0.1347 - precision: 0.9854 - recall: 0.9345 - val_accuracy: 0.9607 - val_auc: 0.9928 - val_loss: 0.0974 - val_precision: 0.9717 - val_recall: 0.9755 - learning_rate: 5.0000e-06
Epoch 12/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9480 - auc: 0.9871 - loss: 0.1385 - precision: 0.9849 - recall: 0.9435
Epoch 12: val_auc improved from 0.99279 to 0.99408, saving model to /content/drive/MyDrive/best_model.h5



Epoch 12: finished saving model to /content/drive/MyDrive/best_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 511s 3s/step - accuracy: 0.9415 - auc: 0.9871 - loss: 0.1396 - precision: 0.9847 - recall: 0.9358 - val_accuracy: 0.9616 - val_auc: 0.9941 - val_loss: 0.0905 - val_precision: 0.9706 - val_recall: 0.9781 - learning_rate: 5.0000e-06
Epoch 13/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9508 - auc: 0.9892 - loss: 0.1305 - precision: 0.9873 - recall: 0.9452
Epoch 13: val_auc did not improve from 0.99408
131/131 ━━━━━━━━━━━━━━━━━━━━ 452s 3s/step - accuracy: 0.9518 - auc: 0.9900 - loss: 0.1224 - precision: 0.9892 - recall: 0.9455 - val_accuracy: 0.9578 - val_auc: 0.9910 - val_loss: 0.1073 - val_precision: 0.9716 - val_recall: 0.9716 - learning_rate: 5.0000e-06
Epoch 14/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9507 - auc: 0.9894 - loss: 0.1323 - precision: 0.9841 - recall: 0.9480
Epoch 14: val_auc did not improve from 0.99408
131/131 ━━━━━━━━━━━━━━━━━━━━ 456s 3s/st

In [1]:
model.save('/content/drive/MyDrive/pneumonia_final_model.h5')
print("Model saved successfully — safe to go!")

NameError: name 'model' is not defined